# NYCU Big Data Assignment 2 — November 7 revision
**Ella Tso / 113700058. Deadline: November 21, 23:55 Taiwan. Forecast: November 22–29.**

This notebook runs the official GH Archive preprocessing before modeling. It does not reuse the retired tables or checkpoints. The source host rejected downloads from the assistant runtime (HTTP403); no new real-data results or submission were fabricated. Run this notebook in your own Colab runtime. Downloads cover a substantial period and may take significant time; no GPU is needed.

The optional storage optimization retains only the **exact unmodified official cohort-filter output** per hour before running the unchanged official eight-table builder. Original file hashes and source URLs remain recorded. Choose `full` if you prefer to retain every original gzip file. Missing/truncated files are documented.

Data Source: https://www.gharchive.org/ . Use only the resulting eight tables for the assignment.

In [ ]:
import os, subprocess, pathlib
os.environ['OMP_NUM_THREADS']='1'
os.environ['OPENBLAS_NUM_THREADS']='1'
PROJECT=pathlib.Path('/content/NYCU-BigData-Assignment2')
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','audit-baselines-20261102','https://github.com/ellatso/NYCU-BigData-Assignment2.git',str(PROJECT)],check=True)
os.chdir(PROJECT)
subprocess.run(['git','rev-parse','HEAD'],check=True)
subprocess.run(['python','-m','pip','install','-r','requirements_revised.txt'],check=True)


## Choose source hours
Dates below are UTC; end is exclusive. The default permits the documented October historical experiments. For a final refresh, extend END_UTC to the chosen completed observation cutoff before the November22 forecast. Keep all future test-week hours out of training. Every distinct data period receives a new prepared-data directory.

Colab local files are temporary. Preserve the downloaded run bundle and manifests before the session expires. For resumable downloads across sessions, set WORK to a folder in your own mounted Drive; the notebook does not mount or share Drive automatically.

In [ ]:
import hashlib, json
WORK=pathlib.Path('/content/asg2_work')
START_UTC='2026-09-01T00:00:00+00:00'
END_UTC='2026-11-07T00:00:00+00:00'
STORAGE='cohort'  # exact official filter; use 'full' to retain original hour files
RAW=WORK/('raw_'+hashlib.sha256((START_UTC+STORAGE).encode()).hexdigest()[:10])
RUN_KEY=hashlib.sha256((START_UTC+END_UTC+STORAGE).encode()).hexdigest()[:12]
DATA=WORK/('prepared_'+RUN_KEY)
OUT=WORK/('run_'+RUN_KEY)
WORK.mkdir(parents=True,exist_ok=True)
def run(*args):
    subprocess.run(['python',*map(str,args)],check=True)
run('scripts/prepare_revised.py','download','--raw',RAW,'--start',START_UTC,'--end',END_UTC,'--storage',STORAGE,'--workers',4)


In [ ]:
from asg2.provenance import verify_data
if not (DATA/'provenance.json').exists():
    run('scripts/prepare_revised.py','build','--raw',RAW,'--out',DATA)
verify_data(DATA)
print('Verified new official preprocessing and all table hashes')


## Matched ablations for Q2 and Q3
The fixed protocol compares observed history, recent activity, multiple time windows, recency/trend removal, static relations, received interactions, collaborator removal and the availability clock. The primary dynamic relational prototype is fixed in advance; thresholds use calibration only and the best calibration reference is reported separately. F1 is explanatory evidence and has **zero direct grade weight** under the revision. Coverage limitations stay attached to every result.

In [ ]:
run('-m','asg2.revised','experiment','--data',DATA,'--out',OUT,'--config','config/revised.json')
run('scripts/summarize_revised.py','--data',DATA,'--out',OUT)
import pandas as pd
from IPython.display import display,Image
display(pd.read_csv(OUT/'slide_evidence.csv'))
display(pd.read_csv(OUT/'coverage.csv'))
display(Image(filename=str(OUT/'ablation.png')))


## Generate and independently reproduce the CSV
The current anonymous roster is checked by the pinned official evaluator. Unknown or not-yet-observed test users use a documented calibration prior; their IDs are not dropped. The output is an early forecast when the source cutoff precedes the final deadline. Keep the same data, code, libraries, model configuration and cutoff for reproduction.

The verification rebuilds all eight tables and fits the candidate models again. It can take time; it is part of the reproduction check, not an additional model search.

In [ ]:
run('-m','asg2.revised','predict','--data',DATA,'--out',OUT,'--refit','--student-id','113700058')
run('scripts/reproduce_revised.py','--raw',RAW,'--data',DATA,'--out',OUT,'--student-id','113700058')
print((OUT/'reproduction_checks.json').read_text())
print((OUT/'prediction_protocol.json').read_text())


In [ ]:
import zipfile
BUNDLE=WORK/('Asg2_Revised_Run_'+RUN_KEY+'.zip')
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,'artifacts_revised/'+p.relative_to(OUT).as_posix())
    for p in DATA.rglob('*'):
        if p.is_file():z.write(p,'prepared_data/'+p.relative_to(DATA).as_posix())
print('Save this run bundle:',BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
files.download(str(OUT/'output_113700058.csv'))


## Before E3 submission
- CSV: `output_113700058.csv`, latest anonymous roster, official format check, reproducible from the recorded source files and cutoff.
- English PDF: `Asg2_113700058.pdf`, teacher template, at most16 pages. Replace every pending-results statement with the actual new-run evidence before calling it final.
- Personally recorded MP4: `Asg2_113700058.mp4`,16–20 minutes. AI-generated video is prohibited. Cover the same Q1–Q4 evidence in both the slides and the narration.
- Public source and README acknowledge GH Archive. Keep raw/private data, models and individual predictions out of the public repository.
- Verify the actual E3 receipt; generating these files does not submit them.